# IndoMultiDomain-Core V1 — Final Privacy Clean & Freeze Candidate

Tahap ini:
- meredaksi email dan nomor telepon yang terdeteksi;
- menghitung ulang hash dan split;
- tidak menghapus URL otomatis;
- tidak menghapus soft duplicates secara otomatis;
- membuat laporan kelompok soft duplicate untuk review konservatif;
- menulis frozen candidate terpisah agar Core lama tetap utuh.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
clean_code="\nfrom pathlib import Path\nimport pandas as pd, numpy as np, re, json, hashlib, unicodedata\n\nROOT=Path('/content/drive/MyDrive/IndoMultiDomain')\nCORE=ROOT/'03_harmonized'/'indomultidomain_core_v1.parquet'\nAUD=ROOT/'04_analysis'/'final_audit'\nOUT=ROOT/'04_analysis'/'final_clean'\nOUT.mkdir(parents=True,exist_ok=True)\n\ndf=pd.read_parquet(CORE)\n\nemail_re=re.compile(r'[\\w.+-]+@[\\w.-]+\\.[A-Za-z]{2,}')\nphone_re=re.compile(r'(?<!\\d)(?:\\+?62|0)8\\d{7,12}(?!\\d)')\n\ndef redact_pii(t):\n    t=str(t)\n    t=email_re.sub('[EMAIL_REDACTED]',t)\n    t=phone_re.sub('[PHONE_REDACTED]',t)\n    return t\n\n# preserve a change log without storing the sensitive substring itself\nmask_email=df.text.astype(str).str.contains(email_re,na=False)\nmask_phone=df.text.astype(str).str.contains(phone_re,na=False)\nchanged=mask_email | mask_phone\n\nchange_log=df.loc[changed,['imd_id','source_id','domain','genre']].copy()\nchange_log['email_redacted']=mask_email.loc[changed].values\nchange_log['phone_redacted']=mask_phone.loc[changed].values\nchange_log.to_csv(OUT/'pii_redaction_log.csv',index=False)\n\ndf.loc[changed,'text']=df.loc[changed,'text'].map(redact_pii)\n\n# recompute counts/hashes after redaction\ndef norm_text(x):\n    x=unicodedata.normalize('NFC',str(x))\n    return re.sub(r'\\s+',' ',x).strip()\n\ndef thash(x):\n    return hashlib.sha256(norm_text(x).encode('utf-8')).hexdigest()\n\ndf['char_count']=df.text.str.len()\ndf['word_count']=df.text.str.split().str.len()\ndf['duplicate_group']=df.text.map(thash)\n\n# if redaction creates exact duplicates, retain first deterministically\nredaction_dups=df[df.duplicated('duplicate_group',keep=False)].sort_values(['duplicate_group','imd_id'])\nredaction_dups.to_csv(OUT/'post_redaction_duplicate_report.csv',index=False)\ndf=df.sort_values('imd_id').drop_duplicates('duplicate_group',keep='first').reset_index(drop=True)\n\ndef split_hash(h):\n    n=int(h[:8],16)%100\n    return 'train' if n<80 else ('validation' if n<90 else 'test')\ndf['split']=df.duplicate_group.map(split_hash)\n\n# soft normalized duplicate review\ndef soft_norm(t):\n    t=unicodedata.normalize('NFC',str(t)).lower()\n    t=re.sub(r'https?://\\S+',' ',t)\n    t=re.sub(r'[^\\w\\s]',' ',t,flags=re.UNICODE)\n    t=re.sub(r'\\s+',' ',t).strip()\n    return t\n\ndf['_soft_norm']=df.text.map(soft_norm)\nsoft=df[(df._soft_norm.str.len()>=20) & df.duplicated('_soft_norm',keep=False)].copy()\n\n# classify soft duplicate groups\ngrp=soft.groupby('_soft_norm').agg(\n    records=('imd_id','count'),\n    sources=('source_id','nunique'),\n    domains=('domain','nunique'),\n    min_words=('word_count','min'),\n    max_words=('word_count','max')\n).reset_index()\n\n# conservative policy:\n# - only flag for review; do NOT drop soft duplicates automatically.\ngrp['review_priority']=np.select(\n    [\n        (grp['sources']>1),\n        (grp['records']>=3),\n        (grp['max_words']>=20)\n    ],\n    ['high','medium','medium'],\n    default='low'\n)\ngrp.to_csv(OUT/'soft_duplicate_group_summary.csv',index=False)\nsoft[['imd_id','source_id','domain','genre','word_count','text','_soft_norm']].to_csv(\n    OUT/'soft_duplicate_candidates_post_redaction.csv',index=False\n)\n\n# remove helper\ndf=df.drop(columns=['_soft_norm'])\n\n# write frozen candidate\nFROZEN=ROOT/'03_harmonized'/'frozen_candidate'\nFROZEN.mkdir(parents=True,exist_ok=True)\ndf.to_parquet(FROZEN/'indomultidomain_core_v1_frozen_candidate.parquet',index=False)\ndf.to_csv(FROZEN/'indomultidomain_core_v1_frozen_candidate.csv',index=False)\nfor s in ['train','validation','test']:\n    df[df.split==s].to_parquet(FROZEN/f'indomultidomain_{s}_v1_frozen_candidate.parquet',index=False)\n\n# summary\nsummary={\n    'records_after_redaction':int(len(df)),\n    'email_records_redacted':int(mask_email.sum()),\n    'phone_records_redacted':int(mask_phone.sum()),\n    'exact_duplicates_created_by_redaction':int(len(redaction_dups)),\n    'soft_duplicate_candidate_records':int(len(soft)),\n    'soft_duplicate_groups':int(len(grp)),\n    'high_priority_soft_duplicate_groups':int((grp.review_priority=='high').sum()),\n    'medium_priority_soft_duplicate_groups':int((grp.review_priority=='medium').sum()),\n    'low_priority_soft_duplicate_groups':int((grp.review_priority=='low').sum()),\n    'exact_hash_unique_after_redaction':bool(df.duplicate_group.is_unique),\n    'imd_id_unique':bool(df.imd_id.is_unique),\n    'empty_text':int(df.text.astype(str).str.strip().eq('').sum())\n}\n(OUT/'FINAL_CLEAN_SUMMARY.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')\nprint(json.dumps(summary,indent=2))\nprint('Frozen candidate:',FROZEN)\n"
from pathlib import Path
p=Path('/content/drive/MyDrive/IndoMultiDomain/06_builder/indomultidomain_final_clean.py')
p.write_text(clean_code,encoding='utf-8')
exec(compile(clean_code,str(p),'exec'))


{
  "records_after_redaction": 69075,
  "email_records_redacted": 1,
  "phone_records_redacted": 5,
  "exact_duplicates_created_by_redaction": 0,
  "soft_duplicate_candidate_records": 720,
  "soft_duplicate_groups": 294,
  "high_priority_soft_duplicate_groups": 74,
  "medium_priority_soft_duplicate_groups": 51,
  "low_priority_soft_duplicate_groups": 169,
  "exact_hash_unique_after_redaction": true,
  "imd_id_unique": true,
  "empty_text": 0
}
Frozen candidate: /content/drive/MyDrive/IndoMultiDomain/03_harmonized/frozen_candidate
